# COGCHECK Analysis Notebook

This notebook loads exported gameplay data, expands round-level summary JSON into flat features, and prepares a baseline-normalized dataset for early modeling.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 1. Load Exported Data

Load the user and attempts exports. The attempts loader repairs both standard CSV exports and older doubly-quoted local copies so the rest of the notebook can stay stable.


In [2]:
import csv
import json
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)


def load_attempts_csv(path="attempts.csv"):
    df = pd.read_csv(path)

    # Some locally saved attempts.csv files are doubly quoted line-by-line.
    # If pandas sees one giant column, unwrap each line and parse it again.
    if len(df.columns) == 1 and "," in str(df.columns[0]):
        with open(path, "r", encoding="utf-8") as f:
            raw_lines = [line.rstrip("") for line in f if line.strip()]
        unwrapped_rows = []
        for line in raw_lines:
            first_pass = next(csv.reader([line]))
            second_pass = next(csv.reader([first_pass[0]]))
            unwrapped_rows.append(second_pass)
        df = pd.DataFrame(unwrapped_rows[1:], columns=unwrapped_rows[0])

    df.columns = [str(c).strip().strip('"') for c in df.columns]

    if "baseline_flag" in df.columns:
        df["baseline_flag"] = (
            df["baseline_flag"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"true": True, "false": False, "1": True, "0": False})
        )

    if "attempt_number" in df.columns:
        df["attempt_number"] = pd.to_numeric(df["attempt_number"], errors="coerce")
    if "duration_ms" in df.columns:
        df["duration_ms"] = pd.to_numeric(df["duration_ms"], errors="coerce")
    if "sleep_hours" in df.columns:
        df["sleep_hours"] = pd.to_numeric(df["sleep_hours"], errors="coerce")
    if "success" in df.columns:
        df["success"] = (
            df["success"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"true": True, "false": False, "1": True, "0": False})
        )

    return df


attempts_data = load_attempts_csv("attempts.csv")
print(attempts_data.head().to_string())
print(attempts_data.columns.tolist())


                                     id                               user_id  baseline_flag  attempt_number  duration_ms  success                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [3]:
# fill missing alcohol/sleep entries for downstream analysis
attempts_data = attempts_data.fillna({'alcohol_status': 'no', 'sleep_hours': 8})
print(attempts_data[['id', 'user_id', 'baseline_flag', 'attempt_number', 'duration_ms', 'success', 'alcohol_status', 'sleep_hours']].head().to_string())
print(f"Loaded {len(attempts_data)} rounds")


                                     id                               user_id  baseline_flag  attempt_number  duration_ms  success alcohol_status  sleep_hours
0  796832aa-35b1-4a5e-aa5a-ce3ca3eb8d65  b30e3d91-f42d-42d8-b7e1-9dbec6da7f4c           True               3         2465     True                         8.0
1  462152cc-6c0e-4715-a6d8-63ff02c95684  b30e3d91-f42d-42d8-b7e1-9dbec6da7f4c          False               2         2531     True             no          8.0
2  a628a4e3-f942-4871-ba05-f279e280888f  b30e3d91-f42d-42d8-b7e1-9dbec6da7f4c          False               4         6835     True             no          8.0
3  64ee384c-24eb-4146-b4d0-9b13b0dc1e3f  b30e3d91-f42d-42d8-b7e1-9dbec6da7f4c          False               6         4601     True             no          8.0
4  5eab9eef-f220-40ba-8ff1-76147cfd9553  b30e3d91-f42d-42d8-b7e1-9dbec6da7f4c          False               8         2846     True             no          8.0
Loaded 170 rounds


## 2. Expand Round Summary Into Features

The `summary` column stores round-level metrics plus ordered per-ball timing. This section expands that JSON into analysis-ready columns.


In [4]:
# Parse summary strings (JSON)
parsed_summary = attempts_data["summary"].fillna("{}").apply(json.loads)

# Build features grouped by completion order

def expand_by_completion_order(summary):
    out = {}

    # keep top-level scalar fields (exclude nested sections)
    for k, v in summary.items():
        if k in ("completion_order", "per_ball"):
            continue
        out[k] = v

    completion_list = summary.get("completion_order", [])
    per_ball = summary.get("per_ball", {})

    for item in completion_list:
        if not isinstance(item, dict):
            continue
        order = item.get("order")
        if order is None:
            continue
        prefix = f"completion_order_{order}"

        for k, v in item.items():
            out[f"{prefix}_{k}"] = v

        ball_id = item.get("ball_id")
        if ball_id and isinstance(per_ball, dict):
            ball = per_ball.get(ball_id, {})
            if isinstance(ball, dict):
                for k, v in ball.items():
                    out[f"{prefix}_{k}"] = v

    return out

summary_features = parsed_summary.apply(expand_by_completion_order).apply(pd.Series)
summary_features = summary_features.rename(columns=lambda c: c.replace("completion_order_", "order"))

overlap_cols = [c for c in summary_features.columns if c in attempts_data.columns]
summary_features = summary_features.drop(columns=overlap_cols)

attempts_with_summary = pd.concat(
    [attempts_data.drop(columns=["summary"]), summary_features],
    axis=1
)

print(summary_features.head())
print(attempts_with_summary.columns.tolist())


   balls_completed  completion_ratio  total_events  active_ball_time_ms  missed_tap_penalty_ms  score_ms  completion_score_ms  score_points  score_mode_at_round  total_taps  empty_taps  order1_order order1_ball_id order1_corner_id  order1_t_ms  order1_spawn_delay_ms  order1_actual_spawn_t_ms  order1_first_touch_ms  order1_completion_t_ms  order1_completion_order  order1_unresolved_time_ms  order2_order order2_ball_id order2_corner_id  order2_t_ms  order2_spawn_delay_ms  order2_actual_spawn_t_ms  order2_first_touch_ms  order2_completion_t_ms  order2_completion_order  order2_unresolved_time_ms  order3_order order3_ball_id order3_corner_id  order3_t_ms  order3_spawn_delay_ms  order3_actual_spawn_t_ms  order3_first_touch_ms  order3_completion_t_ms  order3_completion_order  order3_unresolved_time_ms  order4_order order4_ball_id order4_corner_id  order4_t_ms  order4_spawn_delay_ms  order4_actual_spawn_t_ms  order4_first_touch_ms  order4_completion_t_ms  order4_completion_order  order4_unreso

In [5]:
# Feature selection + engineering (editable)
max_order = 4

# Rate features
attempts_with_summary['taps_per_sec'] = attempts_with_summary['total_taps'] / (attempts_with_summary['duration_ms'] / 1000)
attempts_with_summary['events_per_sec'] = attempts_with_summary['total_events'] / (attempts_with_summary['duration_ms'] / 1000)
attempts_with_summary['empty_tap_rate'] = attempts_with_summary['empty_taps'] / attempts_with_summary['total_taps']

# Corner encoding (order-specific vector of corners)
corner_map = {
    'topLeft': 0,
    'topRight': 1,
    'bottomLeft': 2,
    'bottomRight': 3
}

feature_cols = [
    'balls_completed',
    'completion_ratio',
    'total_events',
    'duration_ms',
    'total_taps',
    'empty_taps',
    'taps_per_sec',
    'events_per_sec',
    'empty_tap_rate'
]

for order in range(1, max_order + 1):
    base = f"order{order}"
    spawn = f"{base}_spawn_delay_ms"
    first = f"{base}_first_touch_ms"
    complete = f"{base}_completion_t_ms"
    corner = f"{base}_corner_id"

    # derived timing features
    attempts_with_summary[f"{base}_touch_latency_ms"] = attempts_with_summary[first] - attempts_with_summary[spawn]
    attempts_with_summary[f"{base}_completion_latency_ms"] = attempts_with_summary[complete] - attempts_with_summary[spawn]
    attempts_with_summary[f"{base}_post_touch_latency_ms"] = attempts_with_summary[complete] - attempts_with_summary[first]

    # corner encoded as numeric
    attempts_with_summary[f"{base}_corner_code"] = attempts_with_summary[corner].map(corner_map)

    feature_cols += [
        spawn, first, complete,
        f"{base}_touch_latency_ms",
        f"{base}_completion_latency_ms",
        f"{base}_post_touch_latency_ms",
        f"{base}_corner_code"
    ]

# Optional: drop rows with zero duration to avoid inf rates
# attempts_with_summary = attempts_with_summary[attempts_with_summary['duration_ms'] > 0]


In [6]:
# Presentation-friendly feature table (name + description)

# basic descriptions for your selected feature set
feature_descriptions = {
    'balls_completed': 'Number of balls completed in the attempt',
    'completion_ratio': 'Completion ratio for the attempt',
    'total_events': 'Total events recorded during the attempt',
    'duration_ms': 'Attempt duration (ms)',
    'total_taps': 'Total taps during the attempt',
    'empty_taps': 'Taps that did not hit a ball',
    'taps_per_sec': 'Tap rate (taps / second)',
    'events_per_sec': 'Event rate (events / second)',
    'empty_tap_rate': 'Empty taps / total taps'
}

for order in range(1, max_order + 1):
    base = f"order{order}"
    feature_descriptions[f"{base}_spawn_delay_ms"] = f"Spawn delay for ball completed in order {order} (ms)"
    feature_descriptions[f"{base}_first_touch_ms"] = f"Time of first touch for ball completed in order {order} (ms)"
    feature_descriptions[f"{base}_completion_t_ms"] = f"Completion time for ball completed in order {order} (ms)"
    feature_descriptions[f"{base}_touch_latency_ms"] = f"First touch - spawn delay for order {order} (ms)"
    feature_descriptions[f"{base}_completion_latency_ms"] = f"Completion - spawn delay for order {order} (ms)"
    feature_descriptions[f"{base}_post_touch_latency_ms"] = f"Completion - first touch for order {order} (ms)"
    feature_descriptions[f"{base}_corner_code"] = f"Corner code for completion order {order} (0=TL,1=TR,2=BL,3=BR)"

feature_schema = pd.DataFrame({
    'feature': feature_cols,
    'description': [feature_descriptions.get(f, '') for f in feature_cols]
})

print(feature_schema)


                         feature                                               description
0                balls_completed                  Number of balls completed in the attempt
1               completion_ratio                          Completion ratio for the attempt
2                   total_events                  Total events recorded during the attempt
3                    duration_ms                                     Attempt duration (ms)
4                     total_taps                             Total taps during the attempt
5                     empty_taps                              Taps that did not hit a ball
6                   taps_per_sec                                  Tap rate (taps / second)
7                 events_per_sec                              Event rate (events / second)
8                 empty_tap_rate                                   Empty taps / total taps
9          order1_spawn_delay_ms            Spawn delay for ball completed in order 1 (ms)

## 3. Targets And Baseline Normalization

Define candidate target labels and normalize numeric features relative to each player's baseline rounds.


In [7]:
# Target variable options
# alcohol_status values: 'no', 'a_couple', 'yes' (plus blanks)

# clean sleep_hours to numeric
attempts_with_summary['sleep_hours'] = pd.to_numeric(attempts_with_summary['sleep_hours'], errors='coerce')

# Option A: drank anything (a_couple or yes)
attempts_with_summary['target_drank_any'] = attempts_with_summary['alcohol_status'].isin(['a_couple', 'yes']).astype(int)

# Option B: drank a lot (yes only)
attempts_with_summary['target_drank_yes'] = (attempts_with_summary['alcohol_status'] == 'yes').astype(int)

# Option C: impaired proxy (drank anything OR sleep < 4 hours)
attempts_with_summary['target_impaired'] = (
    attempts_with_summary['alcohol_status'].isin(['a_couple', 'yes']) |
    (attempts_with_summary['sleep_hours'] < 4)
).astype(int)

# Pick one target to use
TARGET_COL = 'target_impaired'  # change to target_drank_yes or target_impaired

# Baseline normalization per user
# Use baseline_flag == 1 as baseline attempts
baseline_rows = attempts_with_summary[attempts_with_summary['baseline_flag'] == True]

# Choose numeric feature columns (exclude ids, target, labels)
exclude_cols = {
    'id', 'user_id', 'baseline_flag', 'attempt_number', 'summary',
    'alcohol_status', 'sleep_hours',
    'target_drank_any', 'target_drank_yes', 'target_impaired'
}

numeric_cols = feature_cols

# Compute per-user baseline means
baseline_means = baseline_rows.groupby('user_id')[numeric_cols].mean()

# Normalize: value minus baseline mean (centered)
normalized = attempts_with_summary.set_index('user_id')[numeric_cols].subtract(baseline_means, level=0)
normalized.columns = [f"norm_{c}" for c in normalized.columns]

final_df = pd.concat(
    [attempts_with_summary.reset_index(drop=True), normalized.reset_index(drop=True)],
    axis=1
)

print(final_df[[TARGET_COL]].head())


   target_impaired
0                0
1                0
2                0
3                0
4                0


In [8]:
# Preview of the modeling dataset (selected features + target)
preview_cols = feature_cols + [TARGET_COL]
preview_df = final_df[preview_cols].head(10)
print(preview_df)


   balls_completed  completion_ratio  total_events  duration_ms  total_taps  empty_taps  taps_per_sec  events_per_sec  empty_tap_rate  order1_spawn_delay_ms  order1_first_touch_ms  order1_completion_t_ms  order1_touch_latency_ms  order1_completion_latency_ms  order1_post_touch_latency_ms  order1_corner_code  order2_spawn_delay_ms  order2_first_touch_ms  order2_completion_t_ms  order2_touch_latency_ms  order2_completion_latency_ms  order2_post_touch_latency_ms  order2_corner_code  order3_spawn_delay_ms  order3_first_touch_ms  order3_completion_t_ms  order3_touch_latency_ms  order3_completion_latency_ms  order3_post_touch_latency_ms  order3_corner_code  order4_spawn_delay_ms  order4_first_touch_ms  order4_completion_t_ms  order4_touch_latency_ms  order4_completion_latency_ms  order4_post_touch_latency_ms  order4_corner_code  target_impaired
0                4               1.0            43         2465           4           0      1.622718       17.444219             0.0                

In [9]:
# Cleaner display + CSV export for presentation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

preview_df = final_df[preview_cols].head(10)
print(preview_df)

# Export a larger sample for slides
final_df[preview_cols].head(50).to_csv('modeling_preview.csv', index=False)


   balls_completed  completion_ratio  total_events  duration_ms  total_taps  empty_taps  taps_per_sec  events_per_sec  empty_tap_rate  order1_spawn_delay_ms  order1_first_touch_ms  \
0                4               1.0            43         2465           4           0      1.622718       17.444219             0.0                    330                    717   
1                4               1.0            48         2531           5           1      1.975504       18.964836             0.2                     31                    850   
2                4               1.0            88         6835           8           4      1.170446       12.874909             0.5                   1238                   3190   
3                4               1.0            75         4601           4           0      0.869376       16.300804             0.0                     53                   1353   
4                4               1.0            44         2846           5          

In [10]:
# Preview of normalized features (first 10 rows)
normalized_preview_cols = [f"norm_{c}" for c in feature_cols]
normalized_preview_df = final_df[normalized_preview_cols].head(10)
print(normalized_preview_df)


   norm_balls_completed  norm_completion_ratio  norm_total_events  norm_duration_ms  norm_total_taps  norm_empty_taps  norm_taps_per_sec  norm_events_per_sec  norm_empty_tap_rate  \
0                   0.0                    0.0               14.0        -99.333333         0.666667              0.0           0.211666             4.260430            -0.035714   
1                   0.0                    0.0               -2.0        432.666667        -0.333333              0.0          -0.280992            -3.198222             0.017857   
2                   0.0                    0.0              -12.0       -333.333333        -0.333333              0.0           0.069326            -1.062207             0.017857   
3                   0.0                    0.0              -51.0      -1668.333333        -3.333333             -3.0          -0.096115            -3.552998            -0.410714   
4                   0.0                    0.0              -48.0      -1099.333333       

## 4. Simple Baseline Model

This section runs a lightweight baseline model to check whether the engineered features contain useful signal.


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Build X, y
X = final_df[numeric_cols + [f"norm_{c}" for c in numeric_cols]].copy()
y = final_df[TARGET_COL].copy()

# Drop rows with missing target
mask = y.notna()
X = X[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
,    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print('ROC AUC:', roc_auc_score(y_test, proba))


              precision    recall  f1-score   support

           0       0.93      0.86      0.89        29
           1       0.43      0.60      0.50         5

    accuracy                           0.82        34
   macro avg       0.68      0.73      0.70        34
weighted avg       0.85      0.82      0.84        34

ROC AUC: 0.8137931034482759


In [12]:
# Quick look at feature relevance (logistic regression coefficients)
# This uses the trained model from the previous cell.

feature_names = X.columns
coefs = model.named_steps['clf'].coef_[0]

importance = (
    pd.DataFrame({'feature': feature_names, 'coef': coefs})
    .assign(abs_coef=lambda d: d['coef'].abs())
    .sort_values('abs_coef', ascending=False)
)

print(importance.head(20))


                              feature      coef  abs_coef
2                        total_events  1.054046  1.054046
6                        taps_per_sec -1.053574  1.053574
7                      events_per_sec  0.878911  0.878911
66            norm_order3_corner_code  0.821807  0.821807
59            norm_order2_corner_code -0.786099  0.786099
36                 order4_corner_code -0.765989  0.765989
35       order4_post_touch_latency_ms  0.682581  0.682581
29                 order3_corner_code  0.669390  0.669390
51  norm_order1_post_touch_latency_ms  0.554979  0.554979
54         norm_order2_first_touch_ms -0.495091  0.495091
48        norm_order1_completion_t_ms  0.446726  0.446726
65  norm_order3_post_touch_latency_ms -0.428383  0.428383
56       norm_order2_touch_latency_ms -0.389873  0.389873
50  norm_order1_completion_latency_ms  0.389302  0.389302
55        norm_order2_completion_t_ms -0.378167  0.378167
43                  norm_taps_per_sec  0.377183  0.377183
3             